# Week 2 · Session 2 — Concurrency Patterns & the Pipeline

**Programme:** Post Graduate Certificate in Agentic and RAG Systems for AI Engineering · iHUB DivyaSampark, IIT Roorkee

**What this notebook covers:**
- The fake LLM — a quota-free async stand-in with reliable failures
- Pattern 1: `asyncio.gather` (fire many, wait for all)
- Pattern 2: Batching (process in small chunks)
- Pattern 3: Retry with exponential backoff
- The full pipeline — all three patterns in one flow + JSON logging
- Operational baseline — structured logs, secrets, PII warning
- Practice activity — `as_completed` vs `gather`

**No API key needed.** This entire notebook uses the fake LLM — zero quota, reliable failures for teaching retry.

**Time:** ~60–75 minutes.

---
## 1. Setup & the Fake LLM

Some demos need *reliable, on-demand failures* (to show retry) and zero API quota. The `fake_llm.py` module is a drop-in async stand-in: same `Question → Answer` shape as a real call, with a `fail_rate` knob.

In [12]:
import random

random.random()

0.9479174559518521

In [ ]:
[random.random()<0.9 for i in range(20)]

In [4]:
# !pip install --quiet --break-system-packages pydantic

In [18]:
from fake_llm import Question, Answer, fake_ask_llm, FakeLLMError

# Show the interface:
q = Question(text="What is RAG?")
ans = await fake_ask_llm(q)
print(f"Q: {q.text}")
print(f"A: {ans.text}")
print(f"Cost: ${ans.cost_usd}")
print()
print("Same Question → Answer shape as the real API. Swap is a one-line change.")

Q: What is RAG?
A: RAG grounds an LLM's answer in retrieved documents rather than its training data.
Cost: $0.0001

Same Question → Answer shape as the real API. Swap is a one-line change.


In [27]:
# The fail_rate knob — set it to see failures on demand:
try:
    ans = await fake_ask_llm(q, fail_rate=0.5)  # 100% failure
except FakeLLMError as e:
    print(f"FakeLLMError: {e}")
    print("This is what the retry pattern catches.")

FakeLLMError: simulated transient failure: What is RAG?
This is what the retry pattern catches.


---
## 2. Pattern 1 — `asyncio.gather`

Fire many coroutines concurrently, wait for all of them. You saw this with the real API in Session 1; now with the fake LLM so we can see the timing without quota.

In [29]:
import asyncio
import time

questions = [Question(text=t) for t in [
    "What is RAG?",
    "Name three uses of vector databases.",
    "Why might an LLM hallucinate?",
    "What is exponential backoff?",
    "Explain async in plain language.",
]]

# Sequential:
start = time.perf_counter()
for q in questions:
    await fake_ask_llm(q)
seq = time.perf_counter() - start

# Concurrent:
start = time.perf_counter()
await asyncio.gather(*(fake_ask_llm(q) for q in questions))
con = time.perf_counter() - start

print(f"Sequential: {seq:.1f}s")
print(f"Concurrent: {con:.1f}s  (~{seq/con:.1f}× faster)")
print()
print("gather fires all 5, waits for the slowest. Same pattern as Session 1, now quota-free.")

Sequential: 3.8s
Concurrent: 1.2s  (~3.2× faster)

gather fires all 5, waits for the slowest. Same pattern as Session 1, now quota-free.


---
## 3. Pattern 2 — Batching

Don't gather a thousand at once. Rate limits, memory, and cost all argue for processing in **small chunks** with a polite pause between batches.

In [32]:
list(range(0,20))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [33]:
list(range(0,20,3))

[0, 3, 6, 9, 12, 15, 18]

In [ ]:
BATCH_SIZE = 3

async def run_batched(questions, batch_size=BATCH_SIZE):
    results = []
    for i in range(0, len(questions), batch_size):
        chunk = questions[i:i + batch_size]
        print('Questions in Batch ', questions[i : i + batch_size])
        batch_results = await asyncio.gather(*(fake_ask_llm(q) for q in chunk))
        results.extend(batch_results)
        await asyncio.sleep(0.1)  # polite pause between batches
        print(f"  batch {i//batch_size + 1}: {len(chunk)} calls done")
    return results

start = time.perf_counter()
results = await run_batched(questions)
elapsed = time.perf_counter() - start

print(f"\n{len(results)} answers in {elapsed:.1f}s (batch size {BATCH_SIZE})")
print("Batching = gather in small chunks + a pause between them.")

Questions in Batch  [Question(text='What is RAG?'), Question(text='Name three uses of vector databases.'), Question(text='Why might an LLM hallucinate?')]


  batch 1: 3 calls done
Questions in Batch  [Question(text='What is exponential backoff?'), Question(text='Explain async in plain language.')]
  batch 2: 2 calls done

5 answers in 2.0s (batch size 3)
Batching = gather in small chunks + a pause between them.


**🔧 Try this:** Change `BATCH_SIZE` to `1` (fully sequential batches) and then to `5` (all at once). Watch the timing change.

In [9]:
# Your turn — change batch size and re-run:
results = await run_batched(questions, batch_size=5)  # ← try 1, 2, 5
print(f"Got {len(results)} answers")

  batch 1: 5 calls done
Got 5 answers


---
## 4. Pattern 3 — Retry with Exponential Backoff

A flaky call is retried, waiting longer each time: 1s, 2s, 4s. The fake LLM's `fail_rate` knob makes failures reliable and visible.

**Note:** The OpenAI SDK already retries rate-limit errors (429s) automatically. Our custom retry handles *application-level* transient failures beyond the SDK's scope — e.g., a flaky downstream service, a timeout, or a malformed response.

In [55]:
async def ask_with_retry(q: Question, max_retries: int = 4, fail_rate: float = 0.6):
    for attempt in range(max_retries):
        try:
            answer = await fake_ask_llm(q, fail_rate=fail_rate)
            print(f"  attempt {attempt + 1}: success")
            return answer
        except FakeLLMError:   #this is an indication of temporary failure
            wait = 2 ** attempt  # 1, 2, 4, 8 ...
            if attempt == max_retries - 1:
                print(f"  attempt {attempt + 1}: failed — giving up")
                raise
            print(f"  attempt {attempt + 1}: failed — retrying in {wait}s")
            await asyncio.sleep(wait)

print("Retry with backoff (60% fail rate):\n")
q = Question(text="What is exponential backoff?")
try:
    ans = await ask_with_retry(q)
    print(f"\nA: {ans.text}")
except FakeLLMError:
    print("\nAll retries exhausted (unlucky run — re-run the cell)")

Retry with backoff (60% fail rate):



  attempt 1: failed — retrying in 1s
  attempt 2: failed — retrying in 2s
  attempt 3: success

A: Exponential backoff doubles the wait between retries so a flaky API can recover.


**What to notice:** The wait doubles each time (1s, 2s, 4s). This gives a flaky service time to recover. The `for` loop with `2 ** attempt` is the entire pattern — no library needed.

**🔧 Try this:** Change `fail_rate` to `0.2` (mostly succeeds first try) and `0.9` (almost always needs retries). Run each a few times.

In [11]:
# Your turn — change fail_rate:
try:
    ans = await ask_with_retry(Question(text="Test"), fail_rate=0.2)  # ← try 0.2, 0.9
    print(f"A: {ans.text}")
except FakeLLMError:
    print("All retries exhausted")

  attempt 1: failed — retrying in 1s
  attempt 2: success
A: (simulated answer) A grounded, concise response to: Test


---
## 5. The Full Pipeline — All Three Patterns Together

Read questions from a CSV → ask them in batches concurrently → retry each with backoff → validate the answers → emit one JSON log line per call. This is every pattern from the week in one flow.

### 5a. The data models (Pydantic from Session 1)

In [56]:
# Question and Answer are already imported from fake_llm:
print(f"Question fields: {list(Question.model_fields.keys())}")
print(f"Answer fields:   {list(Answer.model_fields.keys())}")
print()
print("Same typed shapes from Session 1 — Pydantic validates at every boundary.")

Question fields: ['text']
Answer fields:   ['question', 'text', 'cost_usd', 'retries']

Same typed shapes from Session 1 — Pydantic validates at every boundary.


### 5b. The JSON logger

In [57]:
import logging, json, sys

log = logging.getLogger("pipeline")
log.setLevel(logging.INFO)
log.handlers.clear()

handler = logging.StreamHandler(sys.stdout)

class JsonFormatter(logging.Formatter):
    def format(self, record):
        return json.dumps({"level": record.levelname, "msg": record.getMessage(),
                           "t": round(time.time(), 3)})

handler.setFormatter(JsonFormatter())
log.addHandler(handler)

log.info("pipeline logger ready")
print()
print("One JSON record per log line — machine-readable from day one.")
print("⚠ In production, log only metadata (latency, status, tokens)")
print("  — never prompt content unless confirmed PII-free (revisited W9, W21).")

{"level": "INFO", "msg": "pipeline logger ready", "t": 1791702442.454}

One JSON record per log line — machine-readable from day one.
⚠ In production, log only metadata (latency, status, tokens)
  — never prompt content unless confirmed PII-free (revisited W9, W21).


### 5c. ask_with_retry (pattern 3)

In [69]:
FAIL_RATE = 0.4

async def pipeline_ask(q: Question, max_retries: int = 4) -> Answer:
    for attempt in range(max_retries):
        try:
            answer = await fake_ask_llm(q, fail_rate=FAIL_RATE)
            log.info(f"asked: {q.text[:40]}")  # ⚠ truncated prompt — metadata only in prod
            answer.retries = attempt
            return answer
        except FakeLLMError:
            if attempt == max_retries - 1:
                log.warning(f"gave up: {q.text[:40]}")
                raise
            log.warning(f"retry {attempt+1} in {2**attempt}s: {q.text[:40]}")
            await asyncio.sleep(2 ** attempt)

### 5d. run_batch (patterns 1 + 2)

In [ ]:
BATCH_SIZE = 3

async def run_pipeline(questions):
    results = []
    for i in range(0, len(questions), BATCH_SIZE):
        chunk = questions[i:i + BATCH_SIZE]
        print(f'Questions in Batch ', questions[i : i + BATCH_SIZE])
        answers = await asyncio.gather(*(pipeline_ask(q) for q in chunk))
        results.extend(answers)
        await asyncio.sleep(0.1)
    return results

### 5e. Load questions and run

In [73]:
import csv
from pathlib import Path

# Load from the CSV file:
with Path("questions.csv").open() as f:
    questions = [Question(text=row["text"]) for row in csv.DictReader(f)]

print(f"Loaded {len(questions)} questions from questions.csv\n")

start = time.perf_counter()
answers = await run_pipeline(questions)
elapsed = time.perf_counter() - start

print(f"\nGot {len(answers)} validated answers in {elapsed:.1f}s "
      f"(batch size {BATCH_SIZE}, fail rate {FAIL_RATE}):\n")
for a in answers:
    print(f"  Q: {a.question[:55]}")
    print(f"  A: {a.text[:70]}  (retries={a.retries})")
    print()

Loaded 5 questions from questions.csv

Questions in Batch  [Question(text='What is retrieval-augmented generation in one sentence?'), Question(text='Name three real-world uses of vector databases.'), Question(text='Why might an LLM produce confident but incorrect answers?')]


{"level": "INFO", "msg": "asked: Name three real-world uses of vector dat", "t": 1791702824.347}
{"level": "WARNING", "msg": "retry 1 in 1s: What is retrieval-augmented generation i", "t": 1791702824.616}
{"level": "INFO", "msg": "asked: Why might an LLM produce confident but i", "t": 1791702824.685}
{"level": "INFO", "msg": "asked: What is retrieval-augmented generation i", "t": 1791702825.937}
Questions in Batch  [Question(text='What is a Pydantic BaseModel and when would you use one?'), Question(text='Explain async and await in plain language.')]
{"level": "INFO", "msg": "asked: What is a Pydantic BaseModel and when wo", "t": 1791702826.602}
{"level": "INFO", "msg": "asked: Explain async and await in plain languag", "t": 1791702826.823}

Got 5 validated answers in 3.1s (batch size 3, fail rate 0.4):

  Q: What is retrieval-augmented generation in one sentence?
  A: (simulated answer) A grounded, concise response to: What is retrieval-  (retries=1)

  Q: Name three real-world uses of

**What to notice:** The JSON log lines show INFO on success and WARNING on retries — with timestamps close together (parallelism) and backoff gaps (retry waits). Every answer is a validated `Answer` object with a `retries` count.

**🔧 Try this:** Change `FAIL_RATE` to `0.6` and re-run — more WARNING lines, longer wall-clock. Or change `BATCH_SIZE` to `1` (sequential) or `5` (all at once).

---
## 6. Operational Baseline

Three disciplines that start this week and carry forward:

**1. Structured JSON logs.** One JSON record per call, machine-readable. These feed the hallucination rate, retrieval hit rate, and cost-per-query KPIs from W7 onward.

**2. Secrets discipline.** The API key lives in the environment, never in code. `client = OpenAI()` reads it automatically. On Vocareum it's pre-set; off Vocareum it goes in `.env`.

**3. Cost awareness.** The `cost_usd` field on `Answer` is a placeholder now — real cost computation arrives in W4 (tiktoken) and W6. The *field* is there so the *habit* is there.

**4. AI coding-assistant habit** (concept, not a demo). Five steps: ask → read the diff → run it → check security → accept. The tool is whatever you prefer; the habit is what we're teaching.

---
## 7. Practice Activity — `as_completed` vs `gather`

**Goal:** Write a variant that uses `asyncio.as_completed` instead of `gather`. With `gather`, results come back in one batch when the slowest finishes. With `as_completed`, each result prints the instant it's ready.

**Time:** ~15 minutes.

In [18]:
async def run_stream(questions, fail_rate=0.0):
    """Like run_pipeline, but yields answers in finish-order, not input-order."""
    
    async def safe_ask(q):
        try:
            return await ask_with_retry(q, fail_rate=fail_rate)
        except FakeLLMError:
            print(f"  gave up: {q.text[:40]}")
            return None
    
    tasks = [safe_ask(q) for q in questions]
    results = []
    for coro in asyncio.as_completed(tasks):  # ← the one change from gather
        ans = await coro
        if ans:
            print(f"  done: {ans.text[:55]}...")  # prints the instant it's ready
            results.append(ans)
    return results

sample = [Question(text=t) for t in [
    "What is RAG in one sentence?",
    "Name three uses of vector databases.",
    "Why might an LLM hallucinate?",
    "Explain async in plain language.",
    "What is exponential backoff?",
]]

print("--- Clean run (fail_rate=0.0) ---\n")
answers = await run_stream(sample, fail_rate=0.0)
print(f"\nReturned {len(answers)} answers")

--- Clean run (fail_rate=0.0) ---

  attempt 1: success
  done: LLMs hallucinate when they produce confident text not g...
  attempt 1: success
  done: Vector databases store embeddings and power semantic se...
  attempt 1: success
  done: Exponential backoff doubles the wait between retries so...
  attempt 1: success
  done: Async lets one program progress many I/O-bound tasks at...
  attempt 1: success
  done: RAG grounds an LLM's answer in retrieved documents rath...

Returned 5 answers


In [19]:
# Now with failures — retried calls arrive LAST:
print("--- With failures (fail_rate=0.4) ---\n")
answers = await run_stream(sample, fail_rate=0.4)
print(f"\nReturned {len(answers)} answers")

--- With failures (fail_rate=0.4) ---

  attempt 1: failed — retrying in 1s
  attempt 1: success
  done: Vector databases store embeddings and power semantic se...
  attempt 1: failed — retrying in 1s
  attempt 1: failed — retrying in 1s
  attempt 1: success
  done: RAG grounds an LLM's answer in retrieved documents rath...
  attempt 2: failed — retrying in 2s
  attempt 2: failed — retrying in 2s
  attempt 2: success
  done: Async lets one program progress many I/O-bound tasks at...
  attempt 3: success
  done: Exponential backoff doubles the wait between retries so...
  attempt 3: failed — retrying in 4s
  attempt 4: failed — giving up
  gave up: Why might an LLM hallucinate?

Returned 4 answers


**What to notice:** With `gather`, results come back in input order when the slowest finishes. With `as_completed`, the fastest arrives first and retried calls arrive last (pushed by their backoff waits).

**When to use each:**
- **`gather`** — when you need all results before the next step (most pipelines)
- **`as_completed`** — when you want to process results as they stream in, or bail early on the first good match

In [20]:
# Reflection (2-3 sentences):
# 
# What did as_completed make visible that gather hides?
# 
# When would you reach for gather, and when for as_completed?
# 
print("(write your reflection above)")

(write your reflection above)


---
## 8. Check Yourself

1. Name the three concurrency patterns from this week.
2. Why do the retry and pipeline demos use a fake LLM instead of the real API?
3. What causes a WARNING line in the pipeline's JSON logs?
4. The OpenAI SDK auto-retries rate-limit errors. When does *our* custom retry add value?

---
*End of Week 2. Next week: FastAPI, streaming, testing, and the testing-vs-evaluation boundary.*